# Feature Stores in MLOps

A **feature store** is a centralized repository for storing, managing, and serving machine learning features. It bridges the gap between data engineering and machine learning, enabling feature reuse, consistency between training and serving, and collaboration across teams.

## What You Will Learn

1. Why feature stores exist and the problems they solve
2. Open-source solutions: Feast, Hopsworks
3. Enterprise platforms: Tecton, Vertex AI Feature Store, SageMaker Feature Store, Databricks Feature Store
4. Real-time platforms: Fennel, Redis-based stores
5. Feature engineering patterns: window aggregations, embeddings, derived features
6. Point-in-time correctness and label leakage prevention
7. Training-serving skew: causes, detection, prevention
8. Feature pipeline patterns and governance

---

## Architecture Overview

```
+------------------+     +-------------------+     +------------------+
|   Data Sources   |     |   Feature Store   |     |   Consumers      |
|                  |     |                   |     |                  |
| - Data Warehouse |---->| +---------------+ |---->| - Training Jobs  |
| - Event Streams  |     | | Offline Store | |     | - Model Serving  |
| - Databases      |     | | (historical)  | |     | - Analytics      |
| - APIs           |     | +---------------+ |     | - Notebooks      |
+------------------+     |                   |     +------------------+
                         | +---------------+ |
                         | | Online Store  | |
                         | | (low-latency) | |
                         | +---------------+ |
                         |                   |
                         | +---------------+ |
                         | | Feature       | |
                         | | Registry      | |
                         | +---------------+ |
                         +-------------------+
```

The feature store has two main storage layers:
- **Offline store**: Historical feature values for training (high throughput, high latency OK)
- **Online store**: Latest feature values for inference (low latency, typically <10ms)


## 1. Why Feature Stores?

### Problems Without a Feature Store

#### 1.1 Feature Reuse
Without a feature store, every team reimplements the same features independently:
- Team A computes `user_30d_purchase_count` one way
- Team B computes the same feature differently
- Results diverge, causing inconsistent model behavior

#### 1.2 Training-Serving Skew
The most dangerous problem: features computed differently at training time vs. serving time.

```
Training Pipeline:                    Serving Pipeline:
  SQL: AVG(purchase_amount)             Python: sum(amounts)/len(amounts)
  WHERE date > NOW() - 30d              (uses different time window)
  
  Result: 42.7                          Result: 38.2  <-- SKEW!
```

This skew degrades model performance silently the model was trained on data that does not match what it sees in production.

#### 1.3 Point-in-Time Correctness
When creating training datasets, you must use only features that were available **at the time of the event**, not future data:

```
Event: User clicked ad at T=10:00

WRONG: Use user's purchase_count as of T=12:00 (data leakage!)
RIGHT: Use user's purchase_count as of T=09:59 (point-in-time correct)
```

Using future data creates label leakage, causing artificially inflated offline metrics that do not generalize to production.

#### 1.4 Feature Sharing Across Teams
A feature store acts as a feature marketplace:
- Data engineers publish features once
- ML engineers discover and reuse them
- No duplication, consistent definitions

### Benefits Summary

| Problem | Without Feature Store | With Feature Store |
|---------|----------------------|--------------------|
| Feature reuse | Duplicated code across teams | Single source of truth |
| Training-serving skew | Common, hard to detect | Prevented by design |
| Point-in-time correctness | Manual, error-prone | Built-in with time travel |
| Feature discovery | Scattered, undocumented | Centralized catalog |
| Feature freshness | Unknown | Monitored SLAs |
| Data lineage | None | Full provenance tracking |


In [1]:
# Setup: Install required packages
# Run this cell first to install dependencies

import subprocess
import sys

def install_package(package):
    """Install a package quietly."""
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', package, '-q'],
        capture_output=True, text=True
    )
    return result.returncode == 0

packages = [
    'feast',
    'pandas',
    'numpy',
    'pyarrow',
    'scikit-learn',
    'matplotlib',
    'seaborn',
]

print('Installing packages...')
for pkg in packages:
    status = install_package(pkg)
    print(f'  {pkg}: {"OK" if status else "FAILED"}')

print('\nImporting core libraries...')
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print('Setup complete!')


Installing packages...


  feast: OK


  pandas: OK


  numpy: OK


  pyarrow: OK


  scikit-learn: OK


  matplotlib: OK


  seaborn: OK

Importing core libraries...


Setup complete!


## 2. Feast Open-Source Feature Store

**Feast** (Feature Store) is the most widely adopted open-source feature store. It provides:
- A unified API for offline (training) and online (serving) feature retrieval
- Point-in-time correct historical queries
- Materialization: pushing features from offline to online store
- Support for multiple offline backends (Parquet, BigQuery, Snowflake, Redshift)
- Support for multiple online backends (Redis, DynamoDB, Firestore, SQLite)

### Core Concepts

```
+----------------+     +------------------+     +-----------------+
|   Entity       |     |   Data Source    |     |  Feature View   |
|                |     |                  |     |                 |
| user_id        |---->| Parquet/BigQuery  |---->| user_features   |
| driver_id      |     | Kafka/Kinesis    |     | driver_features |
| item_id        |     | Push Source      |     | item_features   |
+----------------+     +------------------+     +-----------------+
                                                        |
                                          +-------------+-------------+
                                          |                           |
                                   +------v------+           +--------v------+
                                   | Offline     |           | Online        |
                                   | Store       |           | Store         |
                                   | (Parquet)   |           | (Redis/SQLite)|
                                   +-------------+           +---------------+
```

### Key Components

| Component | Description | Example |
|-----------|-------------|----------|
| `Entity` | Primary key for feature lookup | `user_id`, `driver_id` |
| `DataSource` | Where raw data lives | Parquet file, BigQuery table |
| `FeatureView` | Group of features from one source | `user_stats_fv` |
| `FeatureService` | Logical grouping for a use case | `fraud_detection_fs` |
| `OnDemandFeatureView` | Computed at request time | `user_age_bucket` |

### feature_store.yaml

```yaml
project: my_ml_project
registry: data/registry.db
provider: local

online_store:
  type: redis
  connection_string: localhost:6379

offline_store:
  type: file  # local Parquet files

entity_key_serialization_version: 2
```

For production with GCP:
```yaml
project: my_ml_project
provider: gcp
registry: gs://my-bucket/registry.pb

online_store:
  type: datastore
  project_id: my-gcp-project

offline_store:
  type: bigquery
  dataset: feast_offline
```


In [2]:
# Feast Complete Working Example
# This demonstrates the full Feast workflow with file-based (local) storage

import os
import tempfile
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Create a temporary directory for our feature store
feast_repo_path = tempfile.mkdtemp(prefix='feast_demo_')
data_path = os.path.join(feast_repo_path, 'data')
os.makedirs(data_path, exist_ok=True)

print(f'Feast repo: {feast_repo_path}')

# ============================================================
# Step 1: Create synthetic feature data
# ============================================================
np.random.seed(42)
n_users = 1000
n_timestamps = 5  # 5 historical snapshots per user

records = []
base_time = datetime(2024, 1, 1)

for user_id in range(1, n_users + 1):
    for day_offset in range(n_timestamps):
        ts = base_time + timedelta(days=day_offset * 7)  # weekly snapshots
        records.append({
            'user_id': user_id,
            'event_timestamp': ts,
            'created': ts,
            # User behavioral features
            'total_purchases_30d': np.random.poisson(3),
            'total_spend_30d': round(np.random.exponential(50), 2),
            'avg_session_duration_minutes': round(np.random.gamma(2, 5), 2),
            'days_since_last_login': np.random.randint(0, 30),
            'click_through_rate': round(np.random.beta(2, 5), 4),
            'is_premium_user': np.random.choice([0, 1], p=[0.8, 0.2]),
        })

user_features_df = pd.DataFrame(records)
user_features_df['event_timestamp'] = pd.to_datetime(user_features_df['event_timestamp'])
user_features_df['created'] = pd.to_datetime(user_features_df['created'])

# Save as Parquet (Feast's default offline store format)
parquet_path = os.path.join(data_path, 'user_features.parquet')
user_features_df.to_parquet(parquet_path, index=False)

print(f'Created {len(user_features_df)} feature rows')
print(f'Saved to: {parquet_path}')
print(f'\nSample data:')
user_features_df.head(3)


Feast repo: /tmp/feast_demo_tiiimgkv


Created 5000 feature rows
Saved to: /tmp/feast_demo_tiiimgkv/data/user_features.parquet

Sample data:


,user_id,event_timestamp,created,total_purchases_30d,total_spend_30d,avg_session_duration_minutes,days_since_last_login,click_through_rate,is_premium_user
0,1,2024-01-01,2024-01-01,4,8.48,23.25,1,0.3781,0
1,1,2024-01-08,2024-01-08,2,48.17,5.87,19,0.1773,0
2,1,2024-01-15,2024-01-15,4,57.64,3.33,14,0.3665,0


In [3]:
# Feast: Write feature_store.yaml and feature definitions

import yaml

# Write feature_store.yaml
feature_store_yaml = {
    'project': 'user_behavior_store',
    'registry': os.path.join(data_path, 'registry.db'),
    'provider': 'local',
    'online_store': {
        'type': 'sqlite',
        'path': os.path.join(data_path, 'online_store.db'),
    },
    'offline_store': {
        'type': 'file'
    },
    'entity_key_serialization_version': 2,
}

yaml_path = os.path.join(feast_repo_path, 'feature_store.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(feature_store_yaml, f, default_flow_style=False)

print('feature_store.yaml:')
print('---')
with open(yaml_path) as f:
    print(f.read())

# Write feature definitions (features.py)
features_py = f'''from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64, String

# --- Entity Definition ---
# Entities are the primary keys for feature lookup
user = Entity(
    name="user",
    description="A user in our platform",
    tags={{"team": "ml-platform", "owner": "data-eng"}},
)

# --- Data Source ---
# Points to the raw data (Parquet file in this case)
user_features_source = FileSource(
    name="user_features_source",
    path="{parquet_path}",
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

# --- Feature View ---
# Groups related features together
user_behavior_fv = FeatureView(
    name="user_behavior",
    entities=[user],
    ttl=timedelta(days=30),  # features expire after 30 days
    schema=[
        Field(name="total_purchases_30d", dtype=Int64),
        Field(name="total_spend_30d", dtype=Float32),
        Field(name="avg_session_duration_minutes", dtype=Float32),
        Field(name="days_since_last_login", dtype=Int64),
        Field(name="click_through_rate", dtype=Float32),
        Field(name="is_premium_user", dtype=Int64),
    ],
    source=user_features_source,
    tags={{"team": "ml-platform", "use_case": "recommendation"}},
)
'''

features_path = os.path.join(feast_repo_path, 'features.py')
with open(features_path, 'w') as f:
    f.write(features_py)

print('features.py written successfully')


feature_store.yaml:
---
entity_key_serialization_version: 2
offline_store:
  type: file
online_store:
  path: /tmp/feast_demo_tiiimgkv/data/online_store.db
  type: sqlite
project: user_behavior_store
provider: local
registry: /tmp/feast_demo_tiiimgkv/data/registry.db

features.py written successfully


In [4]:
# Feast: Apply definitions and run historical/online queries

import subprocess
import sys

# Apply feature definitions to registry
result = subprocess.run(
    ['feast', 'apply'],
    cwd=feast_repo_path,
    capture_output=True,
    text=True
)
print('feast apply output:')
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:500])

# Now use the Feast Python API
try:
    from feast import FeatureStore

    # Initialize the feature store
    fs = FeatureStore(repo_path=feast_repo_path)
    print('\nFeature store initialized successfully')
    print('Registered feature views:', [fv.name for fv in fs.list_feature_views()])
    print('Registered entities:', [e.name for e in fs.list_entities()])

    # ============================================================
    # get_historical_features for training dataset creation
    # ============================================================
    # Entity dataframe: which entities at what timestamps
    entity_df = pd.DataFrame({
        'user_id': [1, 2, 3, 100, 200],
        'event_timestamp': [
            datetime(2024, 1, 10),
            datetime(2024, 1, 15),
            datetime(2024, 1, 20),
            datetime(2024, 2, 1),
            datetime(2024, 2, 14),
        ],
        # Target label (what we want to predict)
        'label_clicked': [1, 0, 1, 1, 0],
    })

    print('\n--- get_historical_features (Training) ---')
    print('Entity dataframe (request):')
    print(entity_df.to_string())

    # Fetch historical features point-in-time correct!
    training_df = fs.get_historical_features(
        entity_df=entity_df,
        features=[
            'user_behavior:total_purchases_30d',
            'user_behavior:total_spend_30d',
            'user_behavior:avg_session_duration_minutes',
            'user_behavior:days_since_last_login',
            'user_behavior:click_through_rate',
            'user_behavior:is_premium_user',
        ]
    ).to_df()

    print('\nTraining dataset (point-in-time correct):')
    print(training_df.to_string())

except Exception as e:
    print(f'Feast API error: {e}')
    print('Demonstrating with mock data instead...')
    # Mock training dataset for illustration
    training_df = pd.DataFrame({
        'user_id': [1, 2, 3, 100, 200],
        'event_timestamp': pd.to_datetime(['2024-01-10', '2024-01-15', '2024-01-20', '2024-02-01', '2024-02-14']),
        'label_clicked': [1, 0, 1, 1, 0],
        'total_purchases_30d': [5, 2, 8, 1, 3],
        'total_spend_30d': [120.5, 45.0, 230.8, 22.1, 78.3],
        'avg_session_duration_minutes': [12.3, 4.5, 18.7, 2.1, 9.8],
        'days_since_last_login': [1, 7, 0, 14, 3],
        'click_through_rate': [0.15, 0.08, 0.22, 0.05, 0.12],
        'is_premium_user': [1, 0, 1, 0, 0],
    })
    print('Mock training dataset:')
    print(training_df.to_string())


feast apply output:
No project found in the repository. Using project name user_behavior_store defined in feature_store.yaml
Applying changes for project user_behavior_store
Created project user_behavior_store
Created entity user
Created feature view user_behavior

Created sqlite table user_behavior_store_user_behavior




/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(



Feature store initialized successfully
Registered feature views: ['user_behavior']
Registered entities: ['user']

--- get_historical_features (Training) ---
Entity dataframe (request):
   user_id event_timestamp  label_clicked
0        1      2024-01-10              1
1        2      2024-01-15              0
2        3      2024-01-20              1
3      100      2024-02-01              1
4      200      2024-02-14              0



Training dataset (point-in-time correct):
   user_id           event_timestamp  label_clicked  total_purchases_30d  total_spend_30d  avg_session_duration_minutes  days_since_last_login  click_through_rate  is_premium_user
0        1 2024-01-10 00:00:00+00:00              1                    2             9.34                         38.65                     18              0.2298                0
1        3 2024-01-20 00:00:00+00:00              1                    2            12.62                          5.41                      3              0.0424                0
2        2 2024-01-15 00:00:00+00:00              0                    1            60.39                          8.79                     16              0.3664                0
3      100 2024-02-01 00:00:00+00:00              1                    1            17.24                          8.93                     24              0.5024                0
4      200 2024-02-14 00:00:00+00:00              0      

In [5]:
# Feast: Materialization + Online Feature Serving

try:
    from feast import FeatureStore
    fs = FeatureStore(repo_path=feast_repo_path)

    # ============================================================
    # Materialization: copy features from offline to online store
    # ============================================================
    # This is typically run on a schedule (e.g., every hour)
    print('--- Materializing features to online store ---')
    fs.materialize_incremental(end_date=datetime.now())
    print('Materialization complete!')

    # ============================================================
    # get_online_features for real-time inference
    # ============================================================
    print('\n--- get_online_features (Serving) ---')
    online_response = fs.get_online_features(
        features=[
            'user_behavior:total_purchases_30d',
            'user_behavior:total_spend_30d',
            'user_behavior:click_through_rate',
            'user_behavior:is_premium_user',
        ],
        entity_rows=[
            {'user_id': 1},
            {'user_id': 2},
            {'user_id': 3},
        ]
    )

    online_df = pd.DataFrame(online_response.to_dict())
    print('Online features retrieved:')
    print(online_df.to_string())
    print('\nLatency: ~1-5ms per request (SQLite), ~0.5-2ms (Redis)')

except Exception as e:
    print(f'Online serving demo: {e}')
    print('\nIllustrating online serving workflow:')
    print('''
    # Production pattern for online serving:

    from feast import FeatureStore
    fs = FeatureStore(repo_path="/path/to/feast/repo")

    # Called at inference time (low latency)
    def get_features_for_user(user_id: int) -> dict:
        response = fs.get_online_features(
            features=[
                "user_behavior:total_purchases_30d",
                "user_behavior:click_through_rate",
            ],
            entity_rows=[{"user_id": user_id}]
        )
        return response.to_dict()

    # Materialization schedule (run via cron or Airflow):
    # feast materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")
    ''')

print('\n=== Feast Workflow Summary ===')
print('1. Define entities, data sources, feature views in Python')
print('2. feast apply -> registers definitions in the registry')
print('3. get_historical_features -> point-in-time correct training data')
print('4. materialize_incremental -> push offline features to online store')
print('5. get_online_features -> low-latency feature retrieval at inference')


--- Materializing features to online store ---
Materializing 1 feature views to 2026-06-19 17:10:41+00:00 into the sqlite online store.

user_behavior from 2026-05-20 12:10:41+00:00 to 2026-06-19 17:10:41+00:00:
Online serving demo: The DataFrame from /tmp/feast_demo_tiiimgkv/data/user_features.parquet being materialized must have at least {'user'} columns present, but these were missing: {'user'} 

Illustrating online serving workflow:

    # Production pattern for online serving:

    from feast import FeatureStore
    fs = FeatureStore(repo_path="/path/to/feast/repo")

    # Called at inference time (low latency)
    def get_features_for_user(user_id: int) -> dict:
        response = fs.get_online_features(
            features=[
                "user_behavior:total_purchases_30d",
                "user_behavior:click_through_rate",
            ],
            entity_rows=[{"user_id": user_id}]
        )
        return response.to_dict()

    # Materialization schedule (run via cron 

/home/dell/Desktop/AI_Tasks/Additional_Data/AI/zero-to-ai-engineer/.venv/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


## 3. Tecton Enterprise Feature Platform

**Tecton** is an enterprise-grade managed feature platform built by the creators of Uber's Michelangelo feature store. It extends Feast's concepts with:

### Key Differentiators

1. **On-Demand Features**: Computed at request time using request data + pre-stored features
2. **Feature Freshness SLAs**: Guarantees on how stale features can be
3. **Streaming Features**: Native Spark Streaming / Flink integration
4. **Backfill**: Automatically backfills historical data for new features
5. **Monitoring**: Built-in feature quality monitoring and drift detection

### Feature Source Types

```
+------------------+------------------+------------------+------------------+
| Batch Source     | Stream Source    | Request Source   | On-Demand Source |
|                  |                  |                  |                  |
| Hive/Snowflake/  | Kafka/Kinesis    | HTTP request     | Computed from    |
| Redshift/BQ      | payload          | payload fields   | other features   |
|                  |                  |                  |                  |
| Freshness: hours | Freshness: secs  | Freshness: N/A   | Freshness: N/A   |
+------------------+------------------+------------------+------------------+
```

### Tecton Feature Definition Example

```python
from tecton import batch_feature_view, FilteredSource, Attribute
from tecton.types import Float64, Int64
from datetime import timedelta

@batch_feature_view(
    sources=[FilteredSource(transactions_batch)],
    entities=[user],
    mode='spark_sql',
    batch_schedule=timedelta(hours=1),
    feature_start_time=datetime(2023, 1, 1),
    description='User transaction aggregates',
    tags={'owner': 'fraud-team', 'sla': '1h'},
)
def user_transaction_features(transactions):
    return f'''
        SELECT
            user_id,
            timestamp,
            COUNT(*) OVER (PARTITION BY user_id
                ORDER BY timestamp
                RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND CURRENT ROW
            ) AS txn_count_7d,
            SUM(amount) OVER (PARTITION BY user_id
                ORDER BY timestamp
                RANGE BETWEEN INTERVAL 30 DAYS PRECEDING AND CURRENT ROW
            ) AS txn_sum_30d
        FROM {transactions}
    '''
```

### On-Demand Features

```python
@on_demand_feature_view(
    sources=[user_stats_fv, RequestSource(schema=[Field('transaction_amount', Float64)])],
    mode='python',
    schema=[Field('is_high_value_txn', Int64)],
)
def high_value_transaction(user_stats, request):
    # Computed at request time no pre-storage needed
    avg_spend = user_stats['avg_spend_30d']
    current_amount = request['transaction_amount']
    return {'is_high_value_txn': int(current_amount > 2 * avg_spend)}
```

### Feature Freshness SLAs

```python
@stream_feature_view(
    sources=[FilteredSource(clickstream_stream)],
    entities=[user],
    mode='spark_sql',
    stream_processing_mode=StreamProcessingMode.CONTINUOUS,
    batch_schedule=timedelta(hours=1),
    # SLA: features must not be older than 5 minutes
    online_serving_index=OnlineServingIndex(freshness=timedelta(minutes=5)),
)
def user_realtime_clicks(clicks):
    return f'SELECT user_id, timestamp, COUNT(*) as click_count_5m FROM {clicks}'
```


## 4. Hopsworks Open-Source Feature Store

**Hopsworks** is a full-stack open-source data platform with a built-in feature store. Unlike Feast (which is primarily a feature retrieval layer), Hopsworks includes:

- **Feature Groups**: The core abstraction (analogous to Feast's FeatureView)
- **Feature Validation**: Great Expectations integration for data quality
- **Embedding Feature Groups**: Native support for vector/embedding features
- **Model Registry**: Built-in model versioning and deployment
- **Statistics**: Automatic feature statistics computation

### Architecture

```
+--------------------------------------------------+
|                  Hopsworks Platform              |
|                                                  |
| +------------+  +------------+  +-----------+   |
| | Feature    |  | Training   |  | Model     |   |
| | Store      |  | Datasets   |  | Registry  |   |
| |            |  |            |  |           |   |
| | - Feature  |  | - Point-in |  | - Model   |   |
| |   Groups   |  |   time     |  |   Versions|   |
| | - Embeddings|  |   joins    |  | - Schemas |   |
| | - Validation|  | - Splits   |  | - Metrics |   |
| +------------+  +------------+  +-----------+   |
|                                                  |
| +------------+  +---------------------------+   |
| | HopsFS     |  | Hive Metastore + Spark    |   |
| | (HDFS-     |  | (offline processing)      |   |
| |  compatible)|  +---------------------------+   |
| +------------+                                   |
+--------------------------------------------------+
```

### Feature Groups in Hopsworks

```python
import hsfs  # Hopsworks Feature Store client

# Connect to Hopsworks
connection = hsfs.connection(
    host='my-hopsworks.ai',
    project='fraud_detection',
    api_key_value='my-api-key'
)
fs = connection.get_feature_store()

# Create a feature group
user_fg = fs.get_or_create_feature_group(
    name='user_behavior_features',
    version=1,
    primary_key=['user_id'],
    event_time='event_timestamp',
    online_enabled=True,  # enables online store
    description='User behavioral features for fraud detection',
    statistics_config={
        'enabled': True,
        'histograms': True,
        'correlations': True,
    }
)

# Insert data
user_fg.insert(user_features_df)

# Feature Validation with Great Expectations
expectation_suite = fs.create_expectation_suite(
    name='user_features_validation',
    run_validation=True,
    validation_ingestion_policy='STRICT',  # reject invalid data
)
expectation_suite.add_expectation(
    ge.core.ExpectationConfiguration(
        expectation_type='expect_column_values_to_be_between',
        kwargs={'column': 'click_through_rate', 'min_value': 0, 'max_value': 1}
    )
)

# Embedding Feature Groups (for vector search)
embedding_fg = fs.get_or_create_feature_group(
    name='product_embeddings',
    version=1,
    primary_key=['product_id'],
    embedding_index=EmbeddingIndex(
        features=[EmbeddingFeature('embedding', 128)],  # 128-dim vector
        index_name='product_emb_index',
    )
)

# k-NN search
similar_products = embedding_fg.find_neighbors(
    embedding=[0.1, 0.2, ...],
    k=10
)
```

### Creating Training Datasets

```python
# Join multiple feature groups for training
query = user_fg.select(['total_purchases_30d', 'total_spend_30d']) \
    .join(transaction_fg.select(['avg_txn_amount', 'max_txn_amount'])) \
    .join(device_fg.select(['device_type', 'os_version']))

# Create a feature view (materialized training dataset)
feature_view = fs.get_or_create_feature_view(
    name='fraud_features_v1',
    query=query,
    labels=['is_fraud'],
)

# Get training/test split with point-in-time correct features
X_train, X_test, y_train, y_test = feature_view.train_test_split(
    test_size=0.2,
    description='Initial fraud model training set',
)
```


## 5. Vertex AI Feature Store (GCP)

**Vertex AI Feature Store** is Google Cloud's managed feature store, tightly integrated with BigQuery and the broader Vertex AI ecosystem.

### Architecture

```
BigQuery (Offline Store)
      |
      | Sync (scheduled)
      v
+---------------------+
| Feature Online Store |   <-- Bigtable-backed, low latency
|                      |
| +------------------+ |
| | Feature Group    | |   <-- Maps to a BigQuery table
| | +------------+   | |
| | | Feature    |   | |   <-- A column in that table
| | | View       |   | |
| | +------------+   | |
| +------------------+ |
+---------------------+
         |
         | fetch_feature_values (grpc, <10ms)
         v
    Prediction Service
```

### Key Components

| Component | Description |
|-----------|-------------|
| `FeatureGroup` | Linked to a BigQuery table; groups related features |
| `Feature` | Individual column within a FeatureGroup |
| `FeatureOnlineStore` | Bigtable or Optimized (for vector search) backend |
| `FeatureView` | A synced snapshot of features in the online store |

### Vertex AI Feature Store Code Example

```python
from google.cloud import aiplatform
from google.cloud.aiplatform import Feature, FeatureGroup

aiplatform.init(project='my-project', location='us-central1')

# Step 1: Create FeatureGroup backed by BigQuery table
feature_group = aiplatform.FeatureGroup.create(
    name='user_behavior_features',
    source=aiplatform.utils.FeatureGroupBigQuerySource(
        uri=['bq://my-project.feature_store.user_features'],
        entity_id_columns=['user_id'],
    ),
    labels={'team': 'ml-platform'},
)

# Step 2: Create individual Features
features = [
    feature_group.create_feature(name='total_purchases_30d'),
    feature_group.create_feature(name='total_spend_30d'),
    feature_group.create_feature(name='click_through_rate'),
]

# Step 3: Create Feature Online Store (Bigtable backend)
feature_online_store = aiplatform.FeatureOnlineStore.create_bigtable_store(
    name='user_features_online_store',
    fixed_node_count=1,
)

# Step 4: Create Feature View (syncs BigQuery -> Bigtable)
feature_view = feature_online_store.create_feature_view(
    name='user_behavior_fv',
    feature_registry_source=aiplatform.utils.FeatureViewFeatureRegistrySource(
        features=[(feature_group, ['total_purchases_30d', 'total_spend_30d'])]
    ),
    sync_config=aiplatform.utils.FeatureViewSyncConfig(
        cron='0 * * * *'  # sync hourly
    ),
)

# Step 5: Online serving
response = feature_view.fetch_feature_values(
    data_key=aiplatform.FeatureViewDataKey(key='user_123')
)
```

### Vector Search with Vertex AI Feature Store

```python
# Optimized store supports approximate nearest-neighbor search
embedding_store = aiplatform.FeatureOnlineStore.create_optimized_store(
    name='product_embedding_store',
)

embedding_fv = embedding_store.create_feature_view(
    name='product_embeddings_fv',
    feature_registry_source=...,
    index_config=aiplatform.utils.FeatureViewIndexConfig(
        embedding_column='embedding',
        dimensions=128,
        distance_measure_type='DOT_PRODUCT_DISTANCE',
        algorithm_config=aiplatform.utils.FeatureViewTreeAhConfig(
            leaf_node_embedding_count=1000,
        ),
    ),
)

# k-NN search
neighbors = embedding_fv.search_nearest_entities(
    query=aiplatform.NearestNeighborQuery(
        embedding=my_query_embedding,
        neighbor_count=10,
    )
)
```


## 6. SageMaker Feature Store (AWS)

**Amazon SageMaker Feature Store** is AWS's managed feature store, deeply integrated with the AWS ecosystem.

### Architecture

```
+---------------------------+
|  SageMaker Feature Store  |
|                           |
| +----------+  +--------+  |
| | Online   |  | Offline|  |
| | Store    |  | Store  |  |
| |          |  |        |  |
| | DynamoDB |  | S3     |  |
| | (ms)     |  | Parquet|  |
| +----------+  +--------+  |
|       ^           ^       |
|       |           |       |
| +-----+-----------+-----+ |
| |      PutRecord          | |
| |  (writes to both)       | |
| +-------------------------+ |
+---------------------------+
         |
  AWS Glue Data Catalog
  (auto-registered for Athena)
```

### Key Features

| Feature | Detail |
|---------|--------|
| Online store | DynamoDB-backed, <10ms latency |
| Offline store | S3 Parquet, auto-cataloged in AWS Glue |
| Write mode | Both stores written simultaneously via `PutRecord` |
| Data types | String, Integral, Fractional |
| Versioning | EventTime-based, each record timestamped |
| IAM integration | Fine-grained access control per feature group |
| Athena queries | Query offline store directly with SQL |

### SageMaker Feature Store Code

```python
import boto3
import sagemaker
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.feature_store.feature_definition import (
    FeatureDefinition, FeatureTypeEnum
)

sess = sagemaker.Session()
sm_client = boto3.client('sagemaker', region_name='us-east-1')
featurestore_runtime = boto3.client(
    'sagemaker-featurestore-runtime',
    region_name='us-east-1'
)

# Define feature group
feature_group = FeatureGroup(
    name='user-behavior-features',
    sagemaker_session=sess
)

feature_group.load_feature_definitions(data_frame=user_features_df)

# Create with both online + offline enabled
feature_group.create(
    s3_uri=f's3://my-bucket/feature-store/',
    record_identifier_name='user_id',
    event_time_feature_name='event_timestamp',
    role_arn='arn:aws:iam::123456789:role/SageMakerRole',
    enable_online_store=True,
)

# Ingest features
feature_group.ingest(data_frame=user_features_df, max_workers=3, wait=True)

# Online retrieval
record = featurestore_runtime.get_record(
    FeatureGroupName='user-behavior-features',
    RecordIdentifierValueAsString='user_123',
    FeatureNames=['total_purchases_30d', 'total_spend_30d'],
)

# Offline (Athena) query
query = feature_group.athena_query()
query.run(
    query_string='''
    SELECT user_id, total_purchases_30d, total_spend_30d
    FROM "user-behavior-features"
    WHERE event_time >= '2024-01-01'
    ''',
    output_location=f's3://my-bucket/athena-results/',
)
training_df = query.as_dataframe()
```


## 7. Databricks Feature Store

**Databricks Feature Store** (now called Databricks Feature Engineering) is workspace-native and built on top of Delta Lake.

### Key Design Principles

1. **Delta Tables as backend**: Features stored in Unity Catalog Delta tables
2. **MLflow integration**: Training runs automatically log feature table lineage
3. **Feature Lookups**: Declarative feature joining at training and inference time
4. **Unity Catalog**: Fine-grained access control, lineage, and search

### Architecture

```
+-------------------------------------------+
|           Databricks Workspace            |
|                                           |
| +----------------+  +------------------+ |
| | Feature        |  | Model Training   | |
| | Engineering    |  |                  | |
| | Client         |  | MLflow logs:     | |
| |                |  | - feature table  | |
| | write_table()  |  |   lineage        | |
| | read_table()   |  | - lookup keys    | |
| | create_        |  | - model version  | |
| | training_set() |  +------------------+ |
| +----------------+                        |
|         |                                 |
|         v                                 |
| +---------------------------------------+ |
| |  Unity Catalog Delta Tables           | |
| |  catalog.schema.feature_table_name   | |
| +---------------------------------------+ |
+-------------------------------------------+
```

### Databricks Feature Store Code

```python
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
import mlflow

fe = FeatureEngineeringClient()

# Step 1: Create feature table
fe.create_table(
    name='catalog.feature_store.user_behavior',
    primary_keys=['user_id'],
    timestamp_keys=['event_timestamp'],  # enables time-travel
    df=user_features_spark_df,
    description='User behavioral features, updated hourly',
)

# Step 2: Write new data (upsert)
fe.write_table(
    name='catalog.feature_store.user_behavior',
    df=new_features_spark_df,
    mode='merge',  # upsert by primary key
)

# Step 3: Create training dataset with Feature Lookups
# The FeatureLookup handles the point-in-time join automatically
feature_lookups = [
    FeatureLookup(
        table_name='catalog.feature_store.user_behavior',
        feature_names=['total_purchases_30d', 'total_spend_30d', 'click_through_rate'],
        lookup_key='user_id',
        timestamp_lookup_key='event_timestamp',  # for point-in-time
    ),
    FeatureLookup(
        table_name='catalog.feature_store.device_features',
        feature_names=['device_type', 'os_version'],
        lookup_key=['user_id', 'session_id'],
    ),
]

training_set = fe.create_training_set(
    df=labels_df,  # contains user_id, event_timestamp, label
    feature_lookups=feature_lookups,
    label='is_fraud',
    exclude_columns=['event_timestamp'],
)
training_df = training_set.load_df()

# Step 4: Train and log model (feature lineage auto-tracked in MLflow)
with mlflow.start_run():
    model = train_model(training_df)
    # log_model saves feature lookup config alongside the model
    fe.log_model(
        model=model,
        artifact_path='fraud_model',
        flavor=mlflow.sklearn,
        training_set=training_set,
    )

# Step 5: Batch inference with automatic feature hydration
predictions = fe.score_batch(
    model_uri='models:/fraud_detection/Production',
    df=new_users_df,  # only needs user_id
    result_type='double',
)
```


## 8. Fennel Real-Time Feature Platform

**Fennel** is a modern Python-native feature platform designed for real-time ML. It uses a pure Python DSL so feature definitions feel natural and stay in one place.

### Core Concepts

| Concept | Description |
|---------|-------------|
| `@source` | Attaches a data source (webhook, Kafka, S3, Snowflake, …) to a dataset |
| `@dataset` | Declares a typed, versioned data schema |
| `@featureset` | Groups related features for a single entity |
| `@extractor` | A pure Python function that computes features from datasets |
| `@aggregation` | Declares an incremental, window-based aggregation |

### Incremental Computation Model

Fennel re-uses previously computed aggregates and only processes **new** events since the last checkpoint, making real-time aggregations efficient even over large histories.

### Supported Sources

```
Kafka ──────────┐
Webhook ────────┤
S3 / GCS ───────┼──► Fennel Dataset ──► Featureset ──► Online / Offline Store
Snowflake ──────┤
BigQuery ───────┘
```

### Backfill

Because sources carry full history (or a replay mechanism), Fennel can **backfill** offline feature values for training datasets without a separate batch job.

### Code Example Featureset with Extractors

```python
from fennel.datasets import dataset, field
from fennel.featuresets import featureset, feature, extractor
from fennel.sources import source, Kafka
from fennel.aggregations import aggregation, Count, Sum
from datetime import datetime
import pandas as pd

kafka_source = Kafka(
    name="clickstream",
    bootstrap_servers="broker:9092",
    topic="user_clicks",
    group_id="fennel-consumer",
    sasl_mechanism="PLAIN",
)

@source(kafka_source, disorder="2h", cdc="append")
@dataset
class ClickEvent:
    user_id: str = field(key=True)
    item_id: str
    click_ts: datetime = field(timestamp=True)
    duration_sec: float

@dataset
class UserClickStats:
    user_id: str = field(key=True)
    ts: datetime = field(timestamp=True)
    clicks_7d: int
    total_duration_7d: float

    @aggregation
    def aggregate_clicks(cls):
        return [
            Count(window="7d", into_field=str(cls.clicks_7d)),
            Sum(of="duration_sec", window="7d", into_field=str(cls.total_duration_7d)),
        ]

@featureset
class UserFeatures:
    user_id: str = feature(dtype=str)
    clicks_last_7d: int = feature(dtype=int)
    avg_duration_7d: float = feature(dtype=float)
    is_power_user: bool = feature(dtype=bool)

    @extractor(deps=[UserClickStats])
    @inputs(UserFeatures.user_id)
    @outputs(clicks_last_7d, avg_duration_7d, is_power_user)
    def from_click_stats(cls, ts: pd.Series, user_id: pd.Series) -> pd.DataFrame:
        stats, _ = UserClickStats.lookup(ts, user_id=user_id)
        clicks = stats["clicks_7d"].fillna(0).astype(int)
        total_dur = stats["total_duration_7d"].fillna(0.0)
        avg_dur = (total_dur / clicks.replace(0, 1)).round(2)
        is_power = clicks >= 50
        return pd.DataFrame({
            "clicks_last_7d": clicks,
            "avg_duration_7d": avg_dur,
            "is_power_user": is_power,
        })
```

The `@extractor` is a **pure function** Fennel can call it both at training time (historical lookup) and at serving time (real-time lookup) with zero code duplication.

## 9. Redis-Based Online Stores

Redis is the **de facto** choice for online feature stores because of its sub-millisecond P99 latency, rich data structures, and built-in TTL support.

### Data Structure Mapping

| Feature Type | Redis Structure | Why |
|--------------|-----------------|-----|
| Scalar features (int/float/str) | Redis Hash | `HSET`/`HGETALL` fetch all fields atomically |
| JSON document features | Redis JSON (RedisJSON module) | Nested, partial updates |
| Dense vector features | Redis Hash + RedisSearch HNSW | ANN search via HNSW index |
| Time-series | Redis Stream or sorted set | Ordered by timestamp |

### Architecture Diagram

```
+------------------------------------------------------------------+
|                    FEATURE STORE LAYERS                          |
|                                                                  |
|  Batch Pipeline --> Parquet / Delta --> Offline Store            |
|         |                                      |                 |
|         |            Materialization           |                 |
|         +--------------------------------------+                 |
|                                                v                 |
|  Streaming Pipeline --------------------> Redis Online Store     |
|                                          +-- Hash (scalars)      |
|  Request --> ML Service --> HGETALL ---- +-- JSON (docs)         |
|              (< 5 ms)      pipeline      +-- HNSW (vectors)      |
+------------------------------------------------------------------+
```

### TTL-Based Expiry

```bash
HSET user:12345 clicks_7d 42 avg_session_min 8.3 tier gold
EXPIRE user:12345 86400   # expire after 24 h
```

### Pipeline Batching

```bash
MULTI
HGETALL user:1001
HGETALL user:1002
HGETALL user:1003
EXEC
```

### Vector Features with RedisSearch

```bash
FT.CREATE idx:user_embeddings
  ON HASH PREFIX 1 emb:
  SCHEMA
    user_id TEXT
    embedding VECTOR HNSW 6
      TYPE FLOAT32
      DIM 128
      DISTANCE_METRIC COSINE

HSET emb:user_42 user_id user_42 embedding <blob_of_512_bytes>

FT.SEARCH idx:user_embeddings
  "*=>[KNN 10 @embedding $vec AS score]"
  PARAMS 2 vec <query_blob>
  SORTBY score
  DIALECT 2
```

### Latency Benchmarks

| Operation | Typical Latency |
|-----------|-----------------|
| Single `HGETALL` (< 20 fields) | 0.2 - 0.5 ms |
| Pipeline of 100 `HGETALL` | 1 - 3 ms |
| HNSW KNN (k=10, 1M vectors) | 1 - 5 ms |
| JSON partial update | 0.3 - 0.8 ms |

In [6]:
import numpy as np
import time
from typing import Any, Dict, List, Optional, Tuple

class RedisFeatureStore:
    """Simulates a Redis-backed online feature store using in-memory dicts."""

    def __init__(self):
        self._store: Dict[str, Dict[str, Any]] = {}
        self._ttl:   Dict[str, float] = {}
        self._vectors: Dict[str, np.ndarray] = {}

    def _is_expired(self, key: str) -> bool:
        exp = self._ttl.get(key)
        return exp is not None and time.time() > exp

    def _clean(self, key: str):
        if self._is_expired(key):
            self._store.pop(key, None)
            self._ttl.pop(key, None)

    def put_features(self, entity_id: str, features: Dict[str, Any], ttl_seconds: Optional[int] = None) -> None:
        key = f"features:{entity_id}"
        self._store[key] = dict(features)
        if ttl_seconds:
            self._ttl[key] = time.time() + ttl_seconds
        print(f"  [PUT] {key} <- {list(features.keys())} (TTL={ttl_seconds}s)")

    def get_features(self, entity_id: str, fields: Optional[List[str]] = None) -> Optional[Dict[str, Any]]:
        key = f"features:{entity_id}"
        self._clean(key)
        data = self._store.get(key)
        if data is None:
            return None
        if fields:
            return {f: data[f] for f in fields if f in data}
        return dict(data)

    def get_batch_features(self, entity_ids: List[str], fields: Optional[List[str]] = None) -> Dict[str, Optional[Dict[str, Any]]]:
        return {eid: self.get_features(eid, fields) for eid in entity_ids}

    def expire_features(self, entity_id: str, ttl_seconds: int) -> None:
        key = f"features:{entity_id}"
        if key in self._store:
            self._ttl[key] = time.time() + ttl_seconds

    def put_vector(self, entity_id: str, vector: np.ndarray) -> None:
        self._vectors[entity_id] = vector / (np.linalg.norm(vector) + 1e-9)

    def get_vector_neighbors(self, query_vector: np.ndarray, k: int = 5) -> List[Tuple[str, float]]:
        if not self._vectors:
            return []
        qv = query_vector / (np.linalg.norm(query_vector) + 1e-9)
        scores = {eid: float(np.dot(qv, vec)) for eid, vec in self._vectors.items()}
        return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:k]

    def stats(self) -> Dict[str, int]:
        live_keys = [k for k in self._store if not self._is_expired(k)]
        return {"total_keys": len(self._store), "live_keys": len(live_keys), "vector_keys": len(self._vectors)}


print("=" * 60)
print("Redis Feature Store Simulation")
print("=" * 60)

store = RedisFeatureStore()

users = {
    "user_001": {"clicks_7d": 42,  "avg_session_min": 8.3,  "tier": "gold",    "age_bucket": "25-34"},
    "user_002": {"clicks_7d": 5,   "avg_session_min": 1.2,  "tier": "bronze",  "age_bucket": "18-24"},
    "user_003": {"clicks_7d": 130, "avg_session_min": 22.7, "tier": "platinum","age_bucket": "35-44"},
}

print("\n--- Writing features ---")
for uid, feats in users.items():
    store.put_features(uid, feats, ttl_seconds=3600)

print("\n--- Single-entity read ---")
print(f"  user_001: {store.get_features('user_001')}")

print("\n--- Partial field read ---")
print(f"  user_002 [clicks_7d, tier]: {store.get_features('user_002', fields=['clicks_7d', 'tier'])}")

print("\n--- Batch read ---")
batch = store.get_batch_features(["user_001", "user_002", "user_003", "user_999"])
for uid, feats in batch.items():
    print(f"  {uid}: {feats if feats else 'MISS'}")

print("\n--- Vector KNN search ---")
np.random.seed(42)
for uid in ["user_001", "user_002", "user_003"]:
    store.put_vector(uid, np.random.rand(16).astype(np.float32))
query_vec = np.random.rand(16).astype(np.float32)
neighbors = store.get_vector_neighbors(query_vec, k=3)
print("  KNN results (entity, cosine_sim):")
for eid, sim in neighbors:
    print(f"    {eid}: {sim:.4f}")

print("\n--- TTL expiry test ---")
store.put_features("temp_user", {"event": "flash_sale"}, ttl_seconds=1)
print(f"  Before: {store.get_features('temp_user')}")
store._ttl["features:temp_user"] = time.time() - 1
print(f"  After:  {store.get_features('temp_user')}")

print(f"\n--- Stats: {store.stats()} ---")

Redis Feature Store Simulation

--- Writing features ---
  [PUT] features:user_001 <- ['clicks_7d', 'avg_session_min', 'tier', 'age_bucket'] (TTL=3600s)
  [PUT] features:user_002 <- ['clicks_7d', 'avg_session_min', 'tier', 'age_bucket'] (TTL=3600s)
  [PUT] features:user_003 <- ['clicks_7d', 'avg_session_min', 'tier', 'age_bucket'] (TTL=3600s)

--- Single-entity read ---
  user_001: {'clicks_7d': 42, 'avg_session_min': 8.3, 'tier': 'gold', 'age_bucket': '25-34'}

--- Partial field read ---
  user_002 [clicks_7d, tier]: {'clicks_7d': 5, 'tier': 'bronze'}

--- Batch read ---
  user_001: {'clicks_7d': 42, 'avg_session_min': 8.3, 'tier': 'gold', 'age_bucket': '25-34'}
  user_002: {'clicks_7d': 5, 'avg_session_min': 1.2, 'tier': 'bronze', 'age_bucket': '18-24'}
  user_003: {'clicks_7d': 130, 'avg_session_min': 22.7, 'tier': 'platinum', 'age_bucket': '35-44'}
  user_999: MISS

--- Vector KNN search ---
  KNN results (entity, cosine_sim):
    user_002: 0.7633
    user_003: 0.7340
    user_001:

## 10. Feature Engineering Patterns

### 10.1 Window Aggregations

Window aggregations compress a raw event stream into fixed-size numeric summaries.

#### Types of Windows

```
Event stream:  --e--e--e--e--e--e--e--e--e--e--e--e--> time

Tumbling (non-overlapping, fixed size):
               [---- W1 ----][---- W2 ----][---- W3 ----]

Sliding (overlapping, fixed size, step < size):
               [------ W1 ------]
                     [------ W2 ------]
                           [------ W3 ------]

Session (event-driven, gap-based):
               [-W1-]   gap   [---- W2 ----]  gap  [-W3-]
```

#### Tumbling Window Formula

$$f(t) = \sum_{i:\, t_i \in [t - w,\; t]} x_i$$

where $w$ is the window width, $t_i$ is the event timestamp, and $x_i$ is the event value.

#### Sliding Window

The window slides by step $s < w$, producing $\lfloor w/s \rfloor$ overlapping windows per period.

#### Session Window

A session ends when the gap between consecutive events exceeds timeout $\delta$. Session length is variable: $|W_{\text{session}}| \leq \text{max\_session\_length}$.

---

### 10.2 Entity Embeddings

High-cardinality categoricals (user IDs, item IDs, zip codes) are mapped to dense vectors:

1. **Co-occurrence graph**: build a bipartite graph (users x items from interaction logs)
2. **Word2Vec-style skip-gram**: treat entity sequences as "sentences"
3. **Output**: each entity -> $d$-dimensional float vector capturing latent similarity

**Benefits over one-hot encoding:**
- Fixed, compact dimension regardless of cardinality
- Semantically similar entities cluster in embedding space
- Works as input to any downstream model

---

### 10.3 Derived Features

| Pattern | Example | Formula |
|---------|---------|---------|
| Ratio | CTR | $\text{clicks} / \text{impressions}$ |
| Interaction | Price x Demand | $f_1 \times f_2$ |
| Lag | Yesterday's revenue | $x_{t-1}$ |
| Log transform | Skewed spend | $\ln(1 + x)$ |
| Bin/Bucket | Age group | $\lfloor x / b \rfloor \cdot b$ |
| Z-score | Feature scaling | $(x - \mu) / \sigma$ |

In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

np.random.seed(0)

n_events = 500
n_users  = 20

events = pd.DataFrame({
    "user_id":    np.random.choice([f"u{i:03d}" for i in range(n_users)], n_events),
    "item_id":    np.random.choice([f"item_{i}" for i in range(50)], n_events),
    "revenue":    np.random.exponential(20, n_events).round(2),
    "duration":   np.random.exponential(5,  n_events).round(2),
    "event_date": pd.to_datetime("2024-01-01") + pd.to_timedelta(
                      np.random.randint(0, 90, n_events), unit="D"),
})
events.sort_values("event_date", inplace=True)
print("Raw events shape:", events.shape)

def compute_rolling_features(df, entity_col, value_col, date_col="event_date", windows=[7, 14, 30]):
    df = df.sort_values([entity_col, date_col]).copy().set_index(date_col)
    records = []
    for entity, grp in df.groupby(entity_col):
        series = grp[value_col].resample("D").sum()
        row = {entity_col: entity}
        for w in windows:
            roll = series.rolling(window=w, min_periods=1)
            row[f"{value_col}_count_{w}d"] = roll.count().iloc[-1]
            row[f"{value_col}_sum_{w}d"]   = roll.sum().iloc[-1].round(2)
            row[f"{value_col}_mean_{w}d"]  = roll.mean().iloc[-1].round(2)
            row[f"{value_col}_std_{w}d"]   = roll.std().iloc[-1].round(2)
            row[f"{value_col}_min_{w}d"]   = roll.min().iloc[-1].round(2)
            row[f"{value_col}_max_{w}d"]   = roll.max().iloc[-1].round(2)
        records.append(row)
    return pd.DataFrame(records)

revenue_features = compute_rolling_features(events, "user_id", "revenue", windows=[7, 14, 30])
print("\nWindow aggregation features (3 rows):")
print(revenue_features.head(3).to_string(index=False))

# Entity embeddings via random projection
EMBEDDING_DIM = 8
all_users = events["user_id"].unique()
all_items = events["item_id"].unique()
user_enc = LabelEncoder().fit(all_users)
item_enc = LabelEncoder().fit(all_items)

interaction_matrix = np.zeros((len(all_users), len(all_items)))
for _, row in events.iterrows():
    i = user_enc.transform([row["user_id"]])[0]
    j = item_enc.transform([row["item_id"]])[0]
    interaction_matrix[i, j] += 1

np.random.seed(42)
proj = np.random.randn(len(all_items), EMBEDDING_DIM) / np.sqrt(EMBEDDING_DIM)
user_embeddings = interaction_matrix @ proj
norms = np.linalg.norm(user_embeddings, axis=1, keepdims=True)
user_embeddings /= (norms + 1e-9)

embed_df = pd.DataFrame(user_embeddings, index=user_enc.classes_,
                        columns=[f"emb_{i}" for i in range(EMBEDDING_DIM)])
embed_df.index.name = "user_id"
print("\nEntity embeddings (3 users, 4 dims):")
print(embed_df.iloc[:3, :4].round(3).to_string())

# Derived features
user_stats = events.groupby("user_id").agg(
    total_revenue=("revenue", "sum"),
    total_events=("revenue", "count"),
    total_duration=("duration", "sum"),
    last_date=("event_date", "max"),
    first_date=("event_date", "min"),
).reset_index()

user_stats["avg_revenue_per_event"] = (user_stats["total_revenue"] / user_stats["total_events"]).round(2)
user_stats["log_total_revenue"] = np.log1p(user_stats["total_revenue"]).round(4)
tenure_days = (user_stats["last_date"] - user_stats["first_date"]).dt.days + 1
user_stats["events_per_day"] = (user_stats["total_events"] / tenure_days).round(3)

print("\nDerived features (4 rows):")
print(user_stats[["user_id","avg_revenue_per_event","log_total_revenue","events_per_day"]].head(4).to_string(index=False))

Raw events shape: (500, 5)

Window aggregation features (3 rows):
user_id  revenue_count_7d  revenue_sum_7d  revenue_mean_7d  revenue_std_7d  revenue_min_7d  revenue_max_7d  revenue_count_14d  revenue_sum_14d  revenue_mean_14d  revenue_std_14d  revenue_min_14d  revenue_max_14d  revenue_count_30d  revenue_sum_30d  revenue_mean_30d  revenue_std_30d  revenue_min_30d  revenue_max_30d
   u000               7.0           55.45             7.92           11.37             0.0           28.66               14.0            84.26              6.02            10.88              0.0            28.81               30.0            97.33              3.24             8.09              0.0            28.81
   u001               7.0           66.54             9.51           16.36             0.0           36.74               14.0            70.27              5.02            12.09              0.0            36.74               30.0            82.82              2.76             8.52              0.0 


Entity embeddings (3 users, 4 dims):
         emb_0  emb_1  emb_2  emb_3
user_id                            
u000     0.027  0.126  0.120 -0.247
u001    -0.181 -0.022 -0.020 -0.375
u002     0.054  0.573 -0.193  0.345

Derived features (4 rows):
user_id  avg_revenue_per_event  log_total_revenue  events_per_day
   u000                  18.88             6.3411           0.366
   u001                  18.70             6.0221           0.247
   u002                  21.78             6.2608           0.276
   u003                  18.00             6.5807           0.471


## 11. Point-in-Time Correct Joins

### The Label Leakage Problem

When building a training dataset, a **naive join** matches each training label to the *latest* available feature values even if those values were computed *after* the label event occurred. This causes the model to "see the future" during training, inflating metrics that will never be reproduced in production.

### As-Of Join Algorithm

For each labelled event $(e, t_e)$, retrieve the feature value **last observed at or before** $t_e$:

$$\text{feature}(e) = \text{last value of } f \text{ where } t_f \leq t_e$$

In pandas this is `pd.merge_asof` (sorted, backward lookup); in SQL it is typically a correlated subquery or window function with `QUALIFY ROW_NUMBER() = 1`.

### Timeline Diagram

```
Feature snapshots:  f1--------f2------------------f3------> time
                    |         |                    |
Label events:       |    e1   |    e2         e3   |   e4
                    |    |    |    |           |   |   |
                    v    v    v    v           v   v   v
Naive join uses:    f1   f3   f2   f3          f3  f3  f3  <- WRONG (leakage)
As-of join uses:    f1   f1   f2   f2          f2  f3  f3  <- CORRECT
```

### Before and After

| Event | Event Time | Naive feature | As-of feature |
|-------|------------|---------------|---------------|
| e1    | 09:00      | f3 (future)   | f1 (correct)  |
| e2    | 11:30      | f3 (future)   | f2 (correct)  |
| e3    | 14:00      | f3 (future)   | f2 (correct)  |
| e4    | 17:45      | f3 (correct)  | f3 (correct)  |

### Practical Considerations

- Feature table must be sorted by `(entity_id, feature_timestamp)` before the as-of merge.
- **Late-arriving data**: set a `tolerance` or `disorder` parameter to exclude features arriving after their validity window.
- **Multiple feature tables**: perform one as-of join per feature group, then horizontally concatenate.
- **Evaluation impact**: label leakage can inflate AUC by 5-30 percentage points depending on feature update frequency and label density.

In [8]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

np.random.seed(7)

N_USERS, N_DAYS, N_EVENTS = 200, 60, 400

user_ids = [f"u{i:04d}" for i in range(N_USERS)]
dates    = pd.date_range("2024-01-01", periods=N_DAYS, freq="D")

feature_rows = []
for uid in user_ids:
    base_score = np.random.uniform(0.1, 0.9)
    drift      = np.random.uniform(-0.003, 0.008)
    for i, d in enumerate(dates[::2]):
        score = float(np.clip(base_score + drift * i + np.random.normal(0, 0.01), 0, 1))
        feature_rows.append({"user_id": uid, "feature_ts": d, "risk_score": score})

features_df = pd.DataFrame(feature_rows).sort_values(["user_id", "feature_ts"]).reset_index(drop=True)

event_rows = []
for _ in range(N_EVENTS):
    uid       = np.random.choice(user_ids)
    event_day = np.random.choice(dates[5:])
    user_feats = features_df[features_df["user_id"] == uid]
    valid      = user_feats[user_feats["feature_ts"] <= event_day]
    true_risk  = valid["risk_score"].iloc[-1] if not valid.empty else 0.5
    label      = int(np.random.rand() < true_risk)
    event_rows.append({"user_id": uid, "event_ts": event_day, "default": label})

events_df = pd.DataFrame(event_rows).sort_values(["user_id", "event_ts"]).reset_index(drop=True)
print("Events:", events_df.shape, "| Features:", features_df.shape)

# WRONG: naive merge (uses latest feature regardless of event time)
latest_features = (
    features_df.sort_values("feature_ts").groupby("user_id").last()
    .reset_index()[["user_id", "risk_score"]].rename(columns={"risk_score": "risk_score_naive"})
)
naive_df = events_df.merge(latest_features, on="user_id", how="left")

# RIGHT: point-in-time correct join
def pit_join(events_df, features_df, entity_col="user_id", event_time_col="event_ts", feature_time_col="feature_ts"):
    ev = events_df.sort_values(event_time_col).reset_index(drop=True)
    ft = features_df.sort_values(feature_time_col).reset_index(drop=True)
    return pd.merge_asof(ev, ft, left_on=event_time_col, right_on=feature_time_col,
                         by=entity_col, direction="backward")

pit_df = pit_join(events_df, features_df).rename(columns={"risk_score": "risk_score_pit"})
print("\nPIT-correct join sample:")
print(pit_df[["user_id","event_ts","feature_ts","default","risk_score_pit"]].head(4).to_string(index=False))

compare_df = pit_df[["user_id","event_ts","default","risk_score_pit"]].copy()
compare_df = compare_df.merge(naive_df[["user_id","event_ts","risk_score_naive"]], on=["user_id","event_ts"], how="left")

def get_auc(df, feature_col, label_col="default"):
    sub = df[[feature_col, label_col]].dropna()
    if sub[label_col].nunique() < 2:
        return float("nan")
    X = sub[[feature_col]].values
    y = sub[label_col].values
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    clf = LogisticRegression()
    clf.fit(X_tr, y_tr)
    return roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])

auc_naive = get_auc(compare_df, "risk_score_naive")
auc_pit   = get_auc(compare_df, "risk_score_pit")
print(f"\nAUC naive (leaky):    {auc_naive:.4f}")
print(f"AUC PIT-correct:      {auc_pit:.4f}")
print(f"Leakage inflation:    +{(auc_naive - auc_pit):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(compare_df["risk_score_naive"].dropna(), bins=25, alpha=0.6, label="Naive (leaky)", color="tomato")
axes[0].hist(compare_df["risk_score_pit"].dropna(), bins=25, alpha=0.6, label="PIT-correct", color="steelblue")
axes[0].set_title("Feature Distribution: Naive vs PIT")
axes[0].set_xlabel("Risk Score"); axes[0].set_ylabel("Count"); axes[0].legend()

bars = axes[1].bar(["Naive (Leaky)", "PIT-Correct"], [auc_naive, auc_pit], color=["tomato", "steelblue"], width=0.4)
for bar, v in zip(bars, [auc_naive, auc_pit]):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.002, f"{v:.4f}", ha="center", va="bottom")
axes[1].set_ylim(0.4, 1.0); axes[1].set_title("Model AUC: Leakage Effect"); axes[1].set_ylabel("ROC-AUC")
plt.tight_layout()
plt.savefig("/tmp/pit_join_auc.png", dpi=100, bbox_inches="tight")
print("\nPlot saved to /tmp/pit_join_auc.png")
plt.show()

Events: (400, 3) | Features: (6000, 3)

PIT-correct join sample:
user_id   event_ts feature_ts  default  risk_score_pit
  u0182 2024-01-06 2024-01-05        1        0.643776
  u0057 2024-01-06 2024-01-05        0        0.502992
  u0053 2024-01-06 2024-01-05        0        0.609959
  u0162 2024-01-06 2024-01-05        0        0.361270

AUC naive (leaky):    0.7107
AUC PIT-correct:      0.7273
Leakage inflation:    +-0.0166



Plot saved to /tmp/pit_join_auc.png


## 12. Training-Serving Skew

**Training-serving skew** occurs when the feature distribution seen at serving time differs from what the model was trained on, silently degrading prediction quality.

### 5 Root Causes

| # | Cause | Example |
|---|-------|---------|
| 1 | **Different preprocessing code** | Training normalises with sklearn; serving normalises with a bespoke SQL expression using slightly different rounding |
| 2 | **Different time windows** | Training uses a 30-day window; a serving bug computes 7 days |
| 3 | **Different null handling** | Training imputes with median; serving passes NULL / 0 / -1 |
| 4 | **Feature drift** | User behaviour changes over time; training distribution no longer reflects reality |
| 5 | **Schema mismatch** | A field is renamed, retyped, or dropped in an upstream table after model deployment |

### Detection Methods

#### Population Stability Index (PSI)

PSI measures how much a distribution has shifted relative to a reference (training) distribution:

$$PSI = \sum_{i=1}^{n} \left(A_i - E_i\right) \ln\!\left(\frac{A_i}{E_i}\right)$$

where $A_i$ is the actual (serving) proportion in bucket $i$ and $E_i$ is the expected (training) proportion.

| PSI Value | Interpretation |
|-----------|----------------|
| < 0.10    | No significant shift |
| 0.10 - 0.20 | Moderate shift monitor |
| > 0.20    | Significant shift retrain |

#### KS Test

The Kolmogorov-Smirnov statistic $D = \sup_x |F_1(x) - F_2(x)|$ measures the maximum difference between two empirical CDFs. A low p-value ($p < 0.05$) indicates a statistically significant shift.

### Prevention Strategies

1. **Feature Store**: single definition used for both training and serving
2. **Schema registry**: enforce schema contracts on every pipeline write
3. **Automated monitoring**: schedule PSI/KS checks after each new serving batch
4. **Shadow mode**: log serving inputs; periodically compare against training distribution
5. **Model versioning**: tie each model artefact to the exact feature pipeline version

In [9]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

N_TRAIN, N_SERVE = 2000, 1000

training_features = pd.DataFrame({
    "age":            np.random.normal(35, 10, N_TRAIN).clip(18, 80),
    "income_k":       np.random.lognormal(3.5, 0.5, N_TRAIN),
    "session_min":    np.random.exponential(8, N_TRAIN),
    "click_rate":     np.random.beta(2, 5, N_TRAIN),
    "days_since_reg": np.random.gamma(5, 20, N_TRAIN),
})

serving_features = pd.DataFrame({
    "age":            np.random.normal(35, 10, N_SERVE).clip(18, 80),
    "income_k":       np.random.lognormal(3.8, 0.6, N_SERVE),   # shifted
    "session_min":    np.random.exponential(12, N_SERVE),        # shifted
    "click_rate":     np.random.beta(2, 5, N_SERVE),
    "days_since_reg": np.random.gamma(8, 25, N_SERVE),           # shifted
})

def compute_psi(expected, actual, buckets=10):
    breakpoints = np.unique(np.percentile(expected, np.linspace(0, 100, buckets + 1)))
    exp_counts, _ = np.histogram(expected, bins=breakpoints)
    act_counts, _ = np.histogram(actual,   bins=breakpoints)
    exp_pct = (exp_counts + 0.0001) / (len(expected) + 0.0001 * len(exp_counts))
    act_pct = (act_counts + 0.0001) / (len(actual)   + 0.0001 * len(act_counts))
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

def detect_skew(training_features, serving_features, psi_threshold=0.10, ks_alpha=0.05):
    results = []
    for col in training_features.columns:
        tr = training_features[col].dropna().values
        sv = serving_features[col].dropna().values
        psi_val          = compute_psi(tr, sv)
        ks_stat, ks_pval = stats.ks_2samp(tr, sv)
        results.append({
            "feature":       col,
            "train_mean":    round(float(tr.mean()), 4),
            "serve_mean":    round(float(sv.mean()), 4),
            "psi":           round(psi_val, 4),
            "ks_stat":       round(ks_stat, 4),
            "ks_pvalue":     round(ks_pval, 4),
            "skew_detected": (psi_val > psi_threshold) or (ks_pval < ks_alpha),
        })
    return pd.DataFrame(results).sort_values("psi", ascending=False)

report = detect_skew(training_features, serving_features)
print("=== Skew Detection Report ===")
print(report.to_string(index=False))
print(f"\nSkewed features: {list(report[report['skew_detected']]['feature'])}")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, feat in zip(axes, ["income_k", "session_min", "days_since_reg", "age"]):
    psi_val = report.loc[report["feature"] == feat, "psi"].values[0]
    flag    = report.loc[report["feature"] == feat, "skew_detected"].values[0]
    ax.hist(training_features[feat], bins=30, alpha=0.55, label="Training", color="steelblue",  density=True)
    ax.hist(serving_features[feat],  bins=30, alpha=0.55, label="Serving",  color="darkorange", density=True)
    ax.set_title(f"{feat}  PSI={psi_val:.3f}  {'SKEW' if flag else 'OK'}", color="red" if flag else "green")
    ax.legend(fontsize=8)
plt.suptitle("Training vs Serving Distributions", fontsize=13)
plt.tight_layout()
plt.savefig("/tmp/skew_detection.png", dpi=100, bbox_inches="tight")
print("\nPlot saved to /tmp/skew_detection.png")
plt.show()

=== Skew Detection Report ===
       feature  train_mean  serve_mean    psi  ks_stat  ks_pvalue  skew_detected
days_since_reg    101.3452    197.0732 2.5116   0.6170     0.0000           True
      income_k     37.4713     54.1823 0.3733   0.2635     0.0000           True
   session_min      7.9474     12.2373 0.1310   0.1630     0.0000           True
    click_rate      0.2864      0.2836 0.0096   0.0240     0.8343          False
           age     35.6006     35.6072 0.0069   0.0165     0.9930          False

Skewed features: ['days_since_reg', 'income_k', 'session_min']



Plot saved to /tmp/skew_detection.png


## 13. Feature Pipeline Patterns

### Pattern 1 Batch Pipeline

```
Scheduler --> Spark/SQL transform --> Parquet/Delta --> Offline Store
                                                              |
                                                        Materialize
                                                              |
                                                              v
                                                       Online Store (Redis)
```

### Pattern 2 Streaming Pipeline

```
Kafka --> Flink/Spark Streaming --> Online Store (Redis, DynamoDB)
                    |
                    +--> Offline Store sink (Parquet / Iceberg)
```

### Pattern 3 On-Demand Pipeline

```
Inference Request --> ML Service
                        1. Fetch stored features (Redis HGETALL)
                        2. Accept request context (cart, query text)
                        3. Compute derived features on-the-fly
                        4. Concatenate -> feature vector
                        5. Model.predict()
```

### Comparison Table

| Dimension        | Batch            | Streaming               | On-Demand              |
|------------------|------------------|-------------------------|------------------------|
| **Latency**      | Minutes-hours    | Seconds                 | Milliseconds           |
| **Freshness**    | Hours-days       | Seconds-minutes         | Request-time           |
| **Throughput**   | Very high        | High                    | Low-medium             |
| **Complexity**   | Low              | Medium                  | Low (if simple logic)  |
| **Cost**         | Low (batch jobs) | Medium (always-on)      | Low (CPU only)         |
| **Best for**     | Historical aggregates, training | Real-time counters, alerts | Request context, ratios |
| **Tools**        | Spark, dbt, SQL  | Flink, Spark Streaming  | Python, REST handler   |

### Lambda Architecture for Features

Most production systems combine all three:
- **Batch layer**: accurate, complete aggregations updated nightly
- **Speed layer**: streaming updates that patch batch values in near-real-time
- **On-demand layer**: request-scoped features that cannot be pre-computed

In [10]:
import pandas as pd
import numpy as np
import os, time
from typing import Dict, Any

np.random.seed(0)

N_USERS, N_EVENTS = 50, 1000
user_ids    = [f"u{i:04d}" for i in range(N_USERS)]
event_dates = pd.date_range("2024-01-01", periods=90, freq="D")

source_df = pd.DataFrame({
    "user_id":    np.random.choice(user_ids, N_EVENTS),
    "item_id":    np.random.choice([f"i{j}" for j in range(100)], N_EVENTS),
    "revenue":    np.random.exponential(30, N_EVENTS).round(2),
    "event_date": np.random.choice(event_dates, N_EVENTS),
})
source_df["event_date"] = pd.to_datetime(source_df["event_date"])
source_df.sort_values("event_date", inplace=True)

_ONLINE_STORE: Dict[str, Dict[str, Any]] = {}


class BatchFeaturePipeline:
    def extract(self, source):
        print(f"[EXTRACT] {len(source):,} rows loaded")
        return source.copy()

    def transform(self, df):
        df = df.sort_values(["user_id", "event_date"]).set_index("event_date")
        records = []
        for uid, grp in df.groupby("user_id"):
            daily = grp["revenue"].resample("D").sum()
            row = {"user_id": uid}
            for w in [7, 30]:
                roll = daily.rolling(w, min_periods=1)
                row[f"revenue_sum_{w}d"]   = round(float(roll.sum().iloc[-1]), 2)
                row[f"revenue_count_{w}d"] = int(roll.count().iloc[-1])
                row[f"revenue_mean_{w}d"]  = round(float(roll.mean().iloc[-1]), 2)
            features = pd.DataFrame(records + [row])
            features["avg_order_value"] = (features["revenue_sum_30d"] / features["revenue_count_30d"].replace(0, 1)).round(2)
            features["log_revenue_30d"] = np.log1p(features["revenue_sum_30d"]).round(4)
            features["is_high_value"]   = features["revenue_sum_30d"] > features["revenue_sum_30d"].quantile(0.75)
            records = features.to_dict("records")
        features = pd.DataFrame(records)
        features["pipeline_ts"] = pd.Timestamp.now()
        print(f"[TRANSFORM] {features.shape} | cols: {list(features.columns)[:6]}...")
        return features

    def load_offline(self, df, path):
        df.to_parquet(path, index=False)
        print(f"[LOAD OFFLINE] {len(df):,} rows -> {path}")
        return path

    def materialize_online(self, offline_path, online_store):
        df = pd.read_parquet(offline_path)
        count = 0
        for _, row in df.iterrows():
            key = f"features:{row['user_id']}"
            online_store[key] = {k: (bool(v) if isinstance(v, (bool, np.bool_)) else
                                     str(v) if isinstance(v, pd.Timestamp) else
                                     float(v) if isinstance(v, (float, np.floating)) else int(v))
                                 for k, v in row.items() if k != "user_id"}
            count += 1
        print(f"[MATERIALIZE] {count} entities in online store")

    def run(self, source, offline_path, online_store):
        raw = self.extract(source)
        features = self.transform(raw)
        self.load_offline(features, offline_path)
        self.materialize_online(offline_path, online_store)
        return features


class StreamingFeaturePipeline:
    def __init__(self, online_store, window_seconds=3600):
        self._online  = online_store
        self._window  = window_seconds
        self._buffers: Dict[str, list] = {}

    def process_event(self, event):
        uid     = event["user_id"]
        ts      = event.get("ts", time.time())
        revenue = float(event.get("revenue", 0))
        buf = self._buffers.setdefault(uid, [])
        buf.append((ts, revenue))
        cutoff = ts - self._window
        self._buffers[uid] = [(t, v) for t, v in buf if t >= cutoff]
        window_revenues = [v for _, v in self._buffers[uid]]
        key = f"stream_features:{uid}"
        self._online[key] = {
            "revenue_sum_1h":   round(sum(window_revenues), 2),
            "revenue_count_1h": len(window_revenues),
            "revenue_max_1h":   round(max(window_revenues), 2) if window_revenues else 0.0,
        }


OFFLINE_PATH = "/tmp/feature_store_batch.parquet"

print("=" * 55)
print("BATCH PIPELINE RUN")
print("=" * 55)
batch_pipe = BatchFeaturePipeline()
features   = batch_pipe.run(source_df, OFFLINE_PATH, _ONLINE_STORE)

sample_key = f"features:{user_ids[0]}"
print(f"\nOnline lookup [{sample_key}]:")
for k, v in list(_ONLINE_STORE.get(sample_key, {}).items())[:5]:
    print(f"  {k}: {v}")

print("\n" + "=" * 55)
print("STREAMING processing 10 events")
print("=" * 55)
stream_pipe = StreamingFeaturePipeline(_ONLINE_STORE, window_seconds=3600)
now = time.time()
for i in range(10):
    ev = {"user_id": np.random.choice(user_ids[:5]),
          "ts":      now - np.random.randint(0, 3600),
          "revenue": round(float(np.random.exponential(20)), 2)}
    stream_pipe.process_event(ev)
    print(f"  Event {i+1}: user={ev['user_id']} revenue=${ev['revenue']}")

print("\nStream feature sample:")
for k, v in list({k: v for k, v in _ONLINE_STORE.items() if k.startswith("stream_")}.items())[:2]:
    print(f"  {k}: {v}")

BATCH PIPELINE RUN
[EXTRACT] 1,000 rows loaded


[TRANSFORM] (50, 11) | cols: ['user_id', 'revenue_sum_7d', 'revenue_count_7d', 'revenue_mean_7d', 'revenue_sum_30d', 'revenue_count_30d']...
[LOAD OFFLINE] 50 rows -> /tmp/feature_store_batch.parquet
[MATERIALIZE] 50 entities in online store

Online lookup [features:u0000]:
  revenue_sum_7d: 65.13
  revenue_count_7d: 7
  revenue_mean_7d: 9.3
  revenue_sum_30d: 107.44
  revenue_count_30d: 30

STREAMING processing 10 events
  Event 1: user=u0002 revenue=$7.63
  Event 2: user=u0003 revenue=$23.99
  Event 3: user=u0004 revenue=$28.76
  Event 4: user=u0004 revenue=$8.83
  Event 5: user=u0000 revenue=$39.67
  Event 6: user=u0002 revenue=$22.39
  Event 7: user=u0004 revenue=$7.82
  Event 8: user=u0001 revenue=$6.87
  Event 9: user=u0004 revenue=$2.03
  Event 10: user=u0001 revenue=$10.53

Stream feature sample:
  stream_features:u0002: {'revenue_sum_1h': 30.02, 'revenue_count_1h': 2, 'revenue_max_1h': 22.39}
  stream_features:u0003: {'revenue_sum_1h': 23.99, 'revenue_count_1h': 1, 'revenue_ma

## 14. Feature Governance

As feature stores grow to hundreds or thousands of features across dozens of teams, governance becomes essential.

### 14.1 Lineage

Feature lineage tracks complete provenance from raw source to model prediction.

```
Raw Source --> Transformation --> Feature View --> Model
(S3 bucket)   (Spark job)        (Feast/Tecton)   (MLflow)
    |               |                  |               |
    +---------------+------------------+---------------+
                    Lineage Graph (tracked in data catalog)
```

When a source table changes schema, lineage lets you immediately identify every downstream feature view and model affected.

### 14.2 Discoverability

- **Feature catalog**: searchable registry of all feature views, descriptions, owners, tags
- **Tagging**: `{domain: finance, entity: user, sensitivity: PII, status: production}`
- **Search**: full-text search over feature names, descriptions, and tags
- **Usage statistics**: which models use each feature, how often it is queried

### 14.3 Documentation

Each feature should carry:
- Human-readable **description** (what does it mean?)
- **Expected range** (`[0, 1]`, `> 0`, categorical: `{A, B, C}`)
- **Update frequency** (hourly, daily, real-time)
- **Owner** and **on-call contact**
- **Known issues** or caveats

### 14.4 Deprecation and Versioning

| Stage | Action |
|-------|--------|
| `experimental` | Feature is being tested; no SLA |
| `production` | Stable; breaking changes require a new version |
| `deprecated` | Scheduled for removal; warn all consumers |
| `archived` | Read-only; no longer computed |

Semantic versioning: `user_revenue_fv:v1` -> `user_revenue_fv:v2` with a migration guide.

### 14.5 Access Control

| Role | Read | Write | Manage |
|------|------|-------|--------|
| Data scientist | own team's features | own team's features | |
| ML engineer | all production features | own team's features | own team's feature views |
| Platform team | all | all | all |
| External auditor | anonymised metadata only | | |

PII features should be tagged and access-controlled separately, with audit logging for every read operation in regulated environments (GDPR, HIPAA).

## 15. Feature Store Comparison

### Comprehensive Platform Comparison

| Platform | Type | Online Store | Offline Store | Streaming | On-Demand | Vector Search | Point-in-Time | Best For |
|----------|------|-------------|---------------|-----------|-----------|---------------|---------------|----------|
| **Feast** | OSS | Redis, DynamoDB, SQLite | Parquet, BigQuery, Redshift | Kafka, Kinesis (contrib) | Yes | Partial (Milvus) | Yes | Full control, no vendor lock-in |
| **Tecton** | Enterprise SaaS | DynamoDB, Redis | S3 + Spark | Yes (native) | Yes | Roadmap | Yes | Enterprises needing managed real-time |
| **Hopsworks** | OSS + Managed | RonDB | Hudi on HDFS/S3 | Kafka + Spark/Flink | Yes | Yes (OpenSearch kNN) | Yes | Full-stack MLOps on-prem or cloud |
| **Vertex AI FS** | GCP Managed | Bigtable | BigQuery | Dataflow | No | No | Yes | GCP-native ML workflows |
| **SageMaker FS** | AWS Managed | DynamoDB | S3 + Glue | Kinesis | No | No | Yes | AWS-native ML pipelines |
| **Databricks FS** | Unity Catalog | Online tables | Delta Lake | DLT | Yes (Python UDFs) | No | Yes | Databricks/Spark workloads |
| **Fennel** | Managed SaaS | Built-in | Built-in | Yes (native) | Yes (extractors) | No | Yes | Python-first teams, rapid iteration |

### Real-Time Capability Comparison

| Platform | Min Serving Latency | Streaming Ingest | Real-Time Aggregations | SLA |
|----------|--------------------|--------------------|----------------------|-----|
| Feast (Redis) | < 1 ms | Via Kafka connector | Manual | Self-managed |
| Tecton | 5-20 ms | Native (first-class) | Yes (time-series windows) | 99.9% managed |
| Hopsworks | 1-5 ms | Kafka + Flink | Yes | 99.5% managed |
| Vertex AI FS | 5-10 ms | Dataflow | Limited | 99.5% GCP SLA |
| SageMaker FS | 5-15 ms | Kinesis | Limited | 99.9% AWS SLA |
| Databricks FS | 10-50 ms | DLT streaming | Via DLT pipelines | 99.9% DBX SLA |
| Fennel | 2-10 ms | Native (webhook/Kafka) | Yes (@aggregation) | Managed SLA |

### Decision Framework

```
Already on GCP?         --> Vertex AI Feature Store
Already on AWS?         --> SageMaker Feature Store
Already on Databricks?  --> Databricks Feature Store
Large enterprise + real-time streaming budget? --> Tecton
Need on-prem / full data sovereignty?          --> Hopsworks OSS
Python-native + startup speed?                 --> Fennel
OSS + control + no SaaS lock-in?               --> Feast
```

In [11]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from typing import Any, Dict, List, Optional
import time

print("=" * 65)
print("MINI FEATURE STORE End-to-End Demo")
print("=" * 65)

np.random.seed(42)

class MiniFeatureStore:
    def __init__(self):
        self._feature_views: Dict[str, dict] = {}
        self._offline:       Dict[str, pd.DataFrame] = {}
        self._online:        Dict[str, Dict[str, Any]] = {}

    def register_feature_view(self, name, entity_col, timestamp_col, feature_cols, description=""):
        self._feature_views[name] = {"entity_col": entity_col, "timestamp_col": timestamp_col,
                                     "feature_cols": feature_cols, "description": description}
        print(f"  [REGISTER] '{name}' | features: {feature_cols}")

    def compute_features(self, view_name, source_df):
        meta = self._feature_views[view_name]
        ec, tc = meta["entity_col"], meta["timestamp_col"]
        feat_df = source_df[[ec, tc] + meta["feature_cols"]].copy()
        self._offline[view_name] = feat_df
        print(f"  [COMPUTE] '{view_name}' -> {feat_df.shape}")
        return feat_df

    def get_training_dataset(self, events, event_entity_col, event_time_col, view_names):
        result = events.sort_values(event_time_col).copy()
        for vn in view_names:
            meta = self._feature_views[vn]
            ec, tc = meta["entity_col"], meta["timestamp_col"]
            feats = self._offline[vn].sort_values(tc)
            result = pd.merge_asof(result, feats.rename(columns={tc: f"{vn}__ts"}),
                                   left_on=event_time_col, right_on=f"{vn}__ts",
                                   by=event_entity_col, direction="backward",
                                   suffixes=("", f"__{vn}"))
        feat_cols = [c for c in result.columns if c not in [event_entity_col, event_time_col, "label"]]
        print(f"  [TRAINING DATASET] shape={result.shape} | features={feat_cols[:6]}...")
        return result

    def materialize_online(self, view_name):
        meta = self._feature_views[view_name]
        ec, tc = meta["entity_col"], meta["timestamp_col"]
        latest = self._offline[view_name].sort_values(tc).groupby(ec).last().reset_index()
        for _, row in latest.iterrows():
            key = str(row[ec])
            self._online.setdefault(key, {}).update({k: v for k, v in row.items() if k not in [ec, tc]})
        print(f"  [MATERIALIZE] '{view_name}' -> {len(latest)} entities online")

    def get_online_features(self, entity_id, feature_names=None):
        data = self._online.get(entity_id, {})
        return {k: data.get(k) for k in feature_names} if feature_names else data

    def monitor_skew(self, training_df, serving_sample, feature_cols):
        rows = []
        for col in feature_cols:
            tr = training_df[col].dropna().values
            sv = serving_sample[col].dropna().values
            if len(tr) < 10 or len(sv) < 10: continue
            ks_stat, ks_pval = stats.ks_2samp(tr, sv)
            rows.append({"feature": col, "train_mean": round(float(tr.mean()), 3),
                         "serve_mean": round(float(sv.mean()), 3),
                         "ks_stat": round(ks_stat, 4), "ks_pval": round(ks_pval, 4),
                         "skew_detected": ks_pval < 0.05})
        return pd.DataFrame(rows)


print("\n[STEP 1] Generate synthetic data")
N_USERS, N_DAYS, N_EVENTS = 100, 60, 300
user_ids   = [f"u{i:04d}" for i in range(N_USERS)]
date_range = pd.date_range("2024-01-01", periods=N_DAYS, freq="D")

feature_rows = []
for uid in user_ids:
    base = np.random.uniform(0.1, 0.9)
    for d in date_range[::3]:
        feature_rows.append({"user_id": uid, "snapshot_ts": d,
                              "spend_30d": round(max(0, np.random.normal(base * 500, 80)), 2),
                              "sessions_30d": max(1, int(np.random.normal(base * 40, 8))),
                              "risk_score": round(float(np.clip(base + np.random.normal(0, 0.05), 0, 1)), 3)})
features_raw = pd.DataFrame(feature_rows)

event_rows = []
for _ in range(N_EVENTS):
    uid  = np.random.choice(user_ids)
    d    = np.random.choice(date_range[5:])
    udata = features_raw[features_raw["user_id"] == uid]
    valid = udata[udata["snapshot_ts"] <= d]
    risk  = float(valid["risk_score"].iloc[-1]) if not valid.empty else 0.5
    event_rows.append({"user_id": uid, "event_ts": d, "label": int(np.random.rand() < risk)})
events_df = pd.DataFrame(event_rows)
print(f"  Features: {features_raw.shape}  |  Events: {events_df.shape}")

print("\n[STEP 2] Register and compute features")
fs = MiniFeatureStore()
fs.register_feature_view("user_spend_fv", "user_id", "snapshot_ts",
                          ["spend_30d", "sessions_30d", "risk_score"],
                          "Monthly spend and engagement signals per user")
fs.compute_features("user_spend_fv", features_raw)

print("\n[STEP 3] Build PIT-correct training dataset")
train_ds = fs.get_training_dataset(events_df, "user_id", "event_ts", ["user_spend_fv"])
train_ds_clean = train_ds.dropna(subset=["spend_30d", "risk_score"])
print(f"  Clean rows: {len(train_ds_clean)} | Label balance: {train_ds_clean['label'].mean():.2f}")

FEAT_COLS = ["spend_30d", "sessions_30d", "risk_score"]
X = train_ds_clean[FEAT_COLS].values
y = train_ds_clean["label"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
clf = LogisticRegression(max_iter=200)
clf.fit(X_tr, y_tr)
auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
print(f"  Logistic Regression AUC: {auc:.4f}")

print("\n[STEP 4] Materialize online and serve")
fs.materialize_online("user_spend_fv")
for uid in user_ids[:3]:
    print(f"  Online features {uid}: {fs.get_online_features(uid, FEAT_COLS)}")

print("\n[STEP 5] Detect training-serving skew")
serving_sample = features_raw.sample(200, random_state=1).copy()
serving_sample["spend_30d"]    *= np.random.uniform(1.0, 1.4, 200)
serving_sample["sessions_30d"] += np.random.randint(0, 10, 200)
skew_report = fs.monitor_skew(train_ds_clean, serving_sample, FEAT_COLS)
print(skew_report.to_string(index=False))

print("\n" + "=" * 65)
print("END-TO-END DEMO COMPLETE")
print("=" * 65)

MINI FEATURE STORE End-to-End Demo

[STEP 1] Generate synthetic data


  Features: (2000, 5)  |  Events: (300, 3)

[STEP 2] Register and compute features
  [REGISTER] 'user_spend_fv' | features: ['spend_30d', 'sessions_30d', 'risk_score']
  [COMPUTE] 'user_spend_fv' -> (2000, 5)

[STEP 3] Build PIT-correct training dataset
  [TRAINING DATASET] shape=(300, 7) | features=['user_spend_fv__ts', 'spend_30d', 'sessions_30d', 'risk_score']...
  Clean rows: 300 | Label balance: 0.48


  Logistic Regression AUC: 0.7635

[STEP 4] Materialize online and serve
  [MATERIALIZE] 'user_spend_fv' -> 100 entities online
  Online features u0000: {'spend_30d': 127.96, 'sessions_30d': 22, 'risk_score': 0.414}
  Online features u0001: {'spend_30d': 414.12, 'sessions_30d': 11, 'risk_score': 0.449}
  Online features u0002: {'spend_30d': 125.54, 'sessions_30d': 1, 'risk_score': 0.171}

[STEP 5] Detect training-serving skew
     feature  train_mean  serve_mean  ks_stat  ks_pval  skew_detected
   spend_30d     242.929     305.149   0.2000   0.0001           True
sessions_30d      19.280      24.035   0.1850   0.0005           True
  risk_score       0.493       0.501   0.0583   0.7936          False

END-TO-END DEMO COMPLETE


## Additional Learning Resources

### Foundational Papers

- **"Meet Michelangelo: Uber's Machine Learning Platform"** (2017) The seminal engineering blog post describing Uber's internal feature store, which popularised the concept of shared, reusable features.
  https://www.uber.com/blog/michelangelo-machine-learning-platform/

- **"Feast: Bridging ML Models and Data"** Technical overview of the Feast open-source feature store, covering design decisions, the registry model, and point-in-time joins.
  https://feast.dev

- **"Scaling Machine Learning at Uber with Michelangelo"** Deep dive into Uber's second-generation architecture, covering online/offline separation and feature reuse at scale.

---

### Official Documentation

| Platform | URL |
|----------|-----|
| Feast | https://docs.feast.dev |
| Tecton | https://docs.tecton.ai |
| Hopsworks | https://docs.hopsworks.ai |
| Vertex AI Feature Store | https://cloud.google.com/vertex-ai/docs/featurestore |
| Amazon SageMaker Feature Store | https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store.html |
| Databricks Feature Store | https://docs.databricks.com/machine-learning/feature-store/index.html |
| Fennel | https://fennel.ai/docs |
| RedisSearch (vector) | https://redis.io/docs/interact/search-and-query/search/vectors/ |

---

### Books and Courses

- **"Designing Machine Learning Systems"** by Chip Huyen Chapter 5 covers feature engineering, feature stores, and training-serving skew in depth.

- **"Machine Learning Systems Design"** (Stanford CS 329S) Covers feature stores in the context of full ML system design. https://stanford-cs329s.github.io

- **"Feature Engineering for Machine Learning"** by Alice Zheng & Amanda Casari (O'Reilly) Practical guide to feature transformation patterns including embeddings, aggregations, and cross-features.

---

### Engineering Blogs

- **Netflix Tech Blog** "Distributed Time Travel for Feature Generation"
  https://netflixtechblog.com/distributed-time-travel-for-feature-generation-389cccdd3907

- **Airbnb Engineering** "Zipline: Airbnb's ML Feature Engineering Platform"
  https://medium.com/airbnb-engineering/zipline-airbnbs-declarative-feature-engineering-framework-d85ca3379421

- **LinkedIn Engineering** "Open Sourcing Feathr: LinkedIn's Feature Store for Productive ML"
  https://engineering.linkedin.com/blog/2022/open-sourcing-feathr

- **DoorDash Engineering** "Building Riviera: A Declarative Real-Time Feature Engineering Framework"
  https://doordash.engineering/2021/03/04/building-a-declarative-real-time-feature-engineering-framework/

---

### Key Takeaways

- **A feature store is not just a key-value cache.** It enforces point-in-time correctness, tracks lineage, and provides a single source of truth for both training and serving.

- **Training-serving skew is the silent killer.** Always use the same feature computation path for training and serving ideally enforced by the feature store itself.

- **Start simple.** Feast with local SQLite + Redis is runnable on a laptop and covers 80% of needs. Add Tecton or Hopsworks when you need managed infrastructure, streaming at scale, or enterprise SLAs.

- **Point-in-time joins are non-negotiable** for time-sensitive ML tasks (fraud, churn, recommendations). Skipping them inflates offline metrics and destroys production performance.

- **Streaming features are valuable but complex.** Build batch features first, then incrementally add streaming where freshness genuinely improves model performance.

- **Governance compounds over time.** Invest early in documentation, tagging, and lineage tracking. The cost of retroactively adding governance to hundreds of features is very high.

- **Vector features are first-class.** Modern feature stores increasingly treat embeddings as native feature types with ANN search support, not afterthoughts stored in blob fields.